# GLADA demo: install, build a model, and run it on audio

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/JakobKauffmann/GLADA/blob/main/notebooks/00_demo/glada_demo.ipynb)

This notebook shows how to install the `glada` package from GitHub, build the
per-group VIB model used in the ICTAI 2026 paper and the MS thesis, run it on a
clip of audio, and score predictions with the package's EER utilities.

**No trained weights are released**, so the model below is untrained and its
spoof scores are random. The goal is to show the API. To train a model, see the
notebooks in `notebooks/04_pgvib_development/` and the setup notes in the
[README](https://github.com/JakobKauffmann/GLADA).

Runs on a free Colab CPU or GPU runtime. The first run downloads
WavLM-Base-Plus (about 380 MB) from Hugging Face.

## 1. Install

In [ ]:
!pip install -q "git+https://github.com/JakobKauffmann/GLADA.git#subdirectory=glada"

## 2. Import and pick a backbone

In [ ]:
import numpy as np
import torch
import torch.nn.functional as F

from glada import BACKBONE_REGISTRY, GLADAConfig, set_seed
from glada.models.glada_model import GLADAModel, PerGroupVIBModel
from glada.evaluation.metrics import compute_eer, compute_eer_with_ci

for key, spec in BACKBONE_REGISTRY.items():
    print(f"{key:18s} {spec['hf_id']:40s} layers={spec['num_layers']:2d} groups={spec['num_groups']}")

## 3. Build the model

The headline runs use WavLM-Base-Plus, whose 13 hidden states (CNN output plus
12 transformer layers) are split into K = 3 layer groups. `PerGroupVIBModel`
puts a VIB on each group. `vib_disabled_groups` makes chosen groups
deterministic, so `(1, 2)` gives the g0-only placement and `()` gives the
all-groups placement.

In [ ]:
set_seed(42)
device = "cuda" if torch.cuda.is_available() else "cpu"

config = GLADAConfig(backbone="wavlm-base-plus", experiment_name="demo")
model = PerGroupVIBModel(config, vib_disabled_groups=(1, 2)).to(device).eval()

n_total = sum(p.numel() for p in model.parameters())
n_head = n_total - sum(p.numel() for p in model.backbone.parameters())
print(f"device={device}  groups={config.num_groups}  hidden_dim={config.hidden_dim}")
print(f"parameters: {n_total/1e6:.1f}M total, {n_head/1e6:.2f}M outside the backbone")

## 4. Load a clip

GLADA expects mono 16 kHz audio and was trained on 4-second crops
(`config.max_audio_len` samples). This uses a short public-domain LibriSpeech
example that ships with librosa. To try your own file, upload it to Colab and
set `path`.

In [ ]:
import librosa

path = librosa.ex("libri1")  # or "/content/your_clip.wav"
wav, sr = librosa.load(path, sr=16000, mono=True)
n = config.max_audio_len
wav = wav[:n] if len(wav) >= n else np.pad(wav, (0, n - len(wav)))
x = torch.from_numpy(wav).float().unsqueeze(0).to(device)  # (batch=1, 64000)
print(x.shape, f"{n / 16000:.1f} s")

## 5. Predict: real or deepfake?

`predict` loads a clip, runs the model, and reports the probability that the
clip is real (bona fide) speech or a deepfake. Index 1 of `spoof_logits` is the
bona fide class.

**The model here is untrained**, so the prediction is close to a coin flip.
With trained weights, the same function gives a real prediction.

In [ ]:
def predict(path, model=model, threshold=0.5):
    wav, _ = librosa.load(path, sr=16000, mono=True)
    n = config.max_audio_len
    wav = wav[:n] if len(wav) >= n else np.pad(wav, (0, n - len(wav)))
    x = torch.from_numpy(wav).float().unsqueeze(0).to(device)

    model.eval()
    with torch.no_grad():
        probs = F.softmax(model(x)["spoof_logits"], dim=1)[0].cpu().numpy()
    p_fake, p_real = float(probs[0]), float(probs[1])

    label = "REAL (bona fide)" if p_real >= threshold else "DEEPFAKE (spoof)"
    bar = lambda p: "#" * round(p * 30) + "-" * (30 - round(p * 30))
    print(f"Clip:        {path.split('/')[-1]}")
    print(f"Prediction:  {label}")
    print(f"  Real       {bar(p_real)}  {p_real:6.1%}")
    print(f"  Deepfake   {bar(p_fake)}  {p_fake:6.1%}")
    return {"label": label, "real": p_real, "deepfake": p_fake}

result = predict(path)

### What the model returns

Besides `spoof_logits`, the output dict holds the bottleneck features, the VIB
KL term, each layer group's pooled representation (`group_outputs`), and the
per-group domain discriminator outputs used during training.

In [ ]:
with torch.no_grad():
    out = model(x)
for k, v in out.items():
    if isinstance(v, torch.Tensor):
        shape = tuple(v.shape)
    elif isinstance(v, (list, tuple)):
        shape = f"{len(v)} x {tuple(v[0].shape)}"
    else:
        shape = type(v).__name__
    print(f"{k:20s} {shape}")

## 6. Compare bottleneck placements

The same constructor builds the other placements studied in the paper.
`GLADAModel` with the default config uses a single VIB after the groups are
pooled.

In [ ]:
variants = {
    "single post-pool VIB": lambda: GLADAModel(config),
    "per-group VIB (all groups)": lambda: PerGroupVIBModel(config, vib_disabled_groups=()),
    "per-group VIB, g0 only": lambda: PerGroupVIBModel(config, vib_disabled_groups=(1, 2)),
}
for name, build in variants.items():
    m = build().to(device).eval()
    with torch.no_grad():
        o = m(x)
    print(f"{name:28s} features {tuple(o['features'].shape)}  kl_loss {float(o['kl_loss']):.3f}")
    del m

## 7. Score with EER

`compute_eer` takes labels (1 = bona fide, 0 = spoof) and bona fide scores,
and returns EER as a percentage.
`compute_eer_with_ci` adds a percentile bootstrap confidence interval, which
is how the thesis reports results. The scores below are synthetic.

In [ ]:
rng = np.random.default_rng(0)
labels = np.r_[np.ones(500), np.zeros(500)]
scores = np.r_[rng.normal(0.7, 0.15, 500), rng.normal(0.4, 0.15, 500)]

print(f"EER = {compute_eer(labels, scores):.2f}%")
print(compute_eer_with_ci(labels, scores, n_bootstrap=200))

## Next steps

- **Train:** `notebooks/04_pgvib_development/` has the runs behind the paper's
  results, and `DATASETS.md` lists the datasets and expected layout.
- **Evaluate across datasets:** `notebooks/05_cross_dataset_eval/`.
- **Read more:** the thesis PDF is in `docs/`.